In [ ]:
# ==========================================
# 0. 安裝與載入套件
# ==========================================
# 請確保已安裝必要套件：

import os  # 作業系統模組：檢查 Excel 檔是否存在、組合路徑
import warnings  # 警告控制：忽略不影響結果的套件警告，讓 Dashboard 輸出更乾淨
import numpy as np  # 數值運算：隨機抽樣、均值/標準差、bootstrap 重抽樣核心
import pandas as pd  # 資料處理：讀 Excel、整理 DataFrame、算管制圖與能力指標
import matplotlib.pyplot as plt  # 繪圖：畫 X-bar/R 管制圖、常態分佈圖
from scipy.stats import norm, shapiro  # norm：常態 pdf/cdf 算 ppm；shapiro：Shapiro-Wilk 常態性檢定
import gradio as gr  # Gradio：快速建立網頁互動式 Dashboard 介面

# 設定 Matplotlib 中文字型與負號顯示
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'Arial Unicode MS', 'SimHei', 'sans-serif']  # 依序嘗試微軟正黑體等，避免中文標題變方框
plt.rcParams['axes.unicode_minus'] = False  # 讓負號正常顯示為 -，而非亂碼方塊

EXCEL_FILE = "factory_production_data.xlsx"  # 即時生產數據檔，與前面模擬器共用同一檔案
SHEET_NAME = "Live_Data"  # 工作表名稱，存放序號/時間/長度/溫度/濕度


# SPC 常數表 (用於子組大小 n=5)
# 若子组大小變更，可擴充此表；本範例子組預設以 5 筆資料為一組
d2_TABLE = {2: 1.128, 3: 1.693, 4: 2.059, 5: 2.326, 6: 2.534}  # d2：由 R-bar 估 sigma 用，sigma = R-bar/d2，key 為子組大小 n
A2_TABLE = {2: 1.880, 3: 1.023, 4: 0.729, 5: 0.577, 6: 0.483}  # A2：X-bar 圖界限係數，UCL/LCL = X-double-bar ± A2*R-bar
D3_TABLE = {2: 0, 3: 0, 4: 0, 5: 0, 6: 0}  # D3：R 圖下界係數，LCL = D3*R-bar (n≤6 時為 0)
D4_TABLE = {2: 3.267, 3: 2.574, 4: 2.282, 5: 2.114, 6: 2.004}  # D4：R 圖上界係數，UCL = D4*R-bar

rng = np.random.default_rng(42)  # 可重現隨機數產生器，種子 42，bootstrap 每次結果相同
BOOTSTRAP_REPS = 200  # Bootstrap 重抽樣次數 200 次，用於估 Cp/Cpk 信賴區間 (教學示範，正式可用1000+)

print("Gradio Dashboard 套件準備完成。")  # 提示：套件載入成功，可繼續往下讀檔建表

In [ ]:
# ==========================================
# 1. 你提供的核心能力指標與 Bootstrap 演算法
# ==========================================
# 本節是 Dashboard 計算引擎：先算 Cp/Cpk/Pp/Ppk，再用 bootstrap 估 95% 信賴區間

def capability_metrics(values, sigma_within, lsl, usl):
    # 功能：計算組內能力(Cp/Cpk)與整體績效(Pp/Ppk)及常態模型預估 ppm
    # values：全部量測值；sigma_within：R-bar/d2 換算的組內 sigma；lsl/usl：規格下/上限
    values = np.asarray(values, dtype=float)  # 轉 float 陣列確保可計算
    mean = values.mean()  # 樣本平均：製程中心位置
    sigma_overall = values.std(ddof=1)  # 整體標準差(n-1無偏估計)，含組間+組內變異

    if sigma_within <= 0 or sigma_overall <= 0:
        # 教學容錯：當資料變異度極小或筆數不足時，賦予微小保護值
        sigma_within = max(sigma_within, 1e-6)  # 防除以零，給極小值讓公式可算不崩潰
        sigma_overall = max(sigma_overall, 1e-6)  # 同上

    cp = (usl - lsl) / (6 * sigma_within)  # Cp：潛在能力，只看寬度不看偏移
    cpu = (usl - mean) / (3 * sigma_within)  # Cpu：上側能力，平均偏高會變小
    cpl = (mean - lsl) / (3 * sigma_within)  # Cpl：下側能力，平均偏低會變小
    cpk = min(cpu, cpl)  # Cpk：取較差側，兼顧偏移與寬度

    pp = (usl - lsl) / (6 * sigma_overall)  # Pp：整體潛能，用整體 sigma 算
    ppu = (usl - mean) / (3 * sigma_overall)  # Ppu：整體上側
    ppl = (mean - lsl) / (3 * sigma_overall)  # Ppl：整體下側
    ppk = min(ppu, ppl)  # Ppk：整體實際能力，通常比 Cpk 小

    p_below = norm.cdf(lsl, loc=mean, scale=sigma_overall)  # P(X<LSL)：常態模型下低於下限機率
    p_above = 1 - norm.cdf(usl, loc=mean, scale=sigma_overall)  # P(X>USL)：高於上限機率

    return {  # 打包回傳供表格/圖使用
        "mean": mean,
        "sigma_within": sigma_within,  # 組內 sigma
        "sigma_overall": sigma_overall,  # 整體 sigma
        "Cp": cp,
        "Cpu": cpu,
        "Cpl": cpl,
        "Cpk": cpk,
        "Pp": pp,
        "Ppu": ppu,
        "Ppl": ppl,
        "Ppk": ppk,
        "ppm_below_lsl": p_below * 1_000_000,  # 下超規 ppm = 機率×百萬
        "ppm_above_usl": p_above * 1_000_000,  # 上超規 ppm
        "ppm_total": (p_below + p_above) * 1_000_000,  # 總不良 ppm
    }


def bootstrap_capability(data_df, lsl, usl, d2, reps=100):
    """以子組為單位重抽樣的 Bootstrap 估算"""
    # 功能：以子組為單位有放回重抽，估 Cp/Cpk/Pp/Ppk 的 95% 信賴區間
    groups = [  # 拆成子組 list，每元素是一組的量測陣列
        group["measurement"].to_numpy(dtype=float)
        for _, group in data_df.groupby("subgroup", observed=True)
    ]

    n_groups = len(groups)  # 子組數
    if n_groups < 2:  # 少於 2 組無法重抽
        return {name: (np.nan, np.nan) for name in ["Cp", "Cpk", "Pp", "Ppk"]}  # 回傳空區間

    samples = {"Cp": [], "Cpk": [], "Pp": [], "Ppk": []}  # 存放每次重抽算出的指標

    for _ in range(reps):  # 重複 reps 次 (預設100，正式建議1000+)
        selected = rng.integers(0, n_groups, size=n_groups)  # 有放回隨機抽 n 組編號
        resampled_groups = [groups[i] for i in selected]  # 取出被抽到的組
        resampled_values = np.concatenate(resampled_groups)  # 併成一維虛擬樣本

        resampled_rbar = np.mean([  # 算本次虛擬樣本的平均全距
            group.max() - group.min()  # 每組全距
            for group in resampled_groups
        ])
        sigma_within = resampled_rbar / d2  # 換算組內 sigma

        if sigma_within <= 0:  # 異常跳過本次
            continue

        result = capability_metrics(resampled_values, sigma_within, lsl, usl)  # 算本次 Cp/Cpk/Pp/Ppk

        for name in samples:  # 存入對應 list
            samples[name].append(result[name])

    intervals = {}  # 存各指標 95% 區間
    for name, estimates in samples.items():
        if len(estimates) < reps * 0.3:  # 成功次數太少 (<30%)，代表資料太不穩
            intervals[name] = (np.nan, np.nan)  # 回傳空區間
        else:
            intervals[name] = tuple(np.percentile(estimates, [2.5, 97.5]))  # percentile 法：取2.5%與97.5%分位為上下限

    return intervals  # 回傳 {Cp:(下,上), Cpk:(下,上), ...}

In [ ]:
# ==========================================
# 1. 你提供的核心能力指標與 Bootstrap 演算法
# ==========================================

def capability_metrics(values, sigma_within, lsl, usl):
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    sigma_overall = values.std(ddof=1)

    if sigma_within <= 0 or sigma_overall <= 0:
        # 教學容錯：當資料變異度極小或筆數不足時，賦予微小保護值
        sigma_within = max(sigma_within, 1e-6)
        sigma_overall = max(sigma_overall, 1e-6)

    cp = (usl - lsl) / (6 * sigma_within)
    cpu = (usl - mean) / (3 * sigma_within)
    cpl = (mean - lsl) / (3 * sigma_within)
    cpk = min(cpu, cpl)

    pp = (usl - lsl) / (6 * sigma_overall)
    ppu = (usl - mean) / (3 * sigma_overall)
    ppl = (mean - lsl) / (3 * sigma_overall)
    ppk = min(ppu, ppl)

    p_below = norm.cdf(lsl, loc=mean, scale=sigma_overall)
    p_above = 1 - norm.cdf(usl, loc=mean, scale=sigma_overall)

    return {
        "mean": mean,
        "sigma_within": sigma_within,
        "sigma_overall": sigma_overall,
        "Cp": cp,
        "Cpu": cpu,
        "Cpl": cpl,
        "Cpk": cpk,
        "Pp": pp,
        "Ppu": ppu,
        "Ppl": ppl,
        "Ppk": ppk,
        "ppm_below_lsl": p_below * 1_000_000,
        "ppm_above_usl": p_above * 1_000_000,
        "ppm_total": (p_below + p_above) * 1_000_000,
    }


def bootstrap_capability(data_df, lsl, usl, d2, reps=100):
    """以子組為單位重抽樣的 Bootstrap 估算"""
    groups = [
        group["measurement"].to_numpy(dtype=float)
        for _, group in data_df.groupby("subgroup", observed=True)
    ]

    n_groups = len(groups)
    if n_groups < 2:
        return {name: (np.nan, np.nan) for name in ["Cp", "Cpk", "Pp", "Ppk"]}

    samples = {"Cp": [], "Cpk": [], "Pp": [], "Ppk": []}

    for _ in range(reps):
        selected = rng.integers(0, n_groups, size=n_groups)
        resampled_groups = [groups[i] for i in selected]
        resampled_values = np.concatenate(resampled_groups)

        resampled_rbar = np.mean([
            group.max() - group.min()
            for group in resampled_groups
        ])
        sigma_within = resampled_rbar / d2

        if sigma_within <= 0:
            continue

        result = capability_metrics(resampled_values, sigma_within, lsl, usl)

        for name in samples:
            samples[name].append(result[name])

    intervals = {}
    for name, estimates in samples.items():
        if len(estimates) < reps * 0.3:
            intervals[name] = (np.nan, np.nan)
        else:
            intervals[name] = tuple(np.percentile(estimates, [2.5, 97.5]))

    return intervals

In [ ]:
# ==========================================
# 2. 儀表板的核心更新函數 (Dashboard Main Logic)
# ==========================================
# Gradio 每次調整參數或按更新都會呼叫此函數，重讀 Excel 並回傳所有圖表/表格/燈號

def update_dashboard(target_qty, lsl_val, usl_val, target_mean, target_sigma, temp_limit, hum_limit):
    # 輸入：目標產量/規格下限/上限/目標均值/目標sigma/溫度上限/濕度上限 (來自 Dashboard 滑桿)
    # 預設回傳預防機制
    empty_fig, _ = plt.subplots(figsize=(1,1))  # 1x1 空圖佔位，讀檔失敗/資料不足時回傳，避免 Gradio 因 None 報錯
    
    if not os.path.exists(EXCEL_FILE):  # Excel 檔還沒被模擬器建立
        return (0, "⚪ 尚未啟動", pd.DataFrame(), pd.DataFrame(), empty_fig, empty_fig, 
                "🔴 尚未讀取數據", empty_fig, "🔴 尚未讀取數據", empty_fig)  # 10 個輸出對應前端元件順序

    try:
        df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME)  # 讀 Live_Data 工作表最新資料
    except Exception as e:  # 檔案被佔用、格式損壞等
        return (0, "⚪ 檔案讀取失敗", pd.DataFrame(), pd.DataFrame(), empty_fig, empty_fig, 
                "🔴 讀取錯誤", empty_fig, "🔴 讀取錯誤", empty_fig)

    if df.empty or len(df) < 3:  # 資料太少無法算統計/管制圖
        return (len(df), "⚪ 資料筆數不足 (需 ≥ 3 筆)", pd.DataFrame(), pd.DataFrame(), 
                empty_fig, empty_fig, "🟢 正常", empty_fig, "🟢 正常", empty_fig)

    # 1. 產量與圓餅圖、警示燈
    current_qty = len(df)  # 目前產量 = 列數，一列代表一件
    progress_pct = min(100.0, (current_qty / max(1, target_qty)) * 100.0)  # 達成率%，max(1,..)防目標0除錯，上限100%
    
    if progress_pct >= 90.0:  # 90% 以上
        alarm_light = f"🚨 警報！已達目標產量 {progress_pct:.1f}% (超過 90%)"  # 紅燈提醒快達標
    else:
        alarm_light = f"🟢 正常運作中 (進度 {progress_pct:.1f}%)"  # 綠燈正常

    # 畫進度圓餅圖
    fig_pie, ax_pie = plt.subplots(figsize=(4, 4))  # 建 4x4 圓餅圖
    remaining_pct = max(0, 100.0 - progress_pct)  # 未完成%，防負值
    ax_pie.pie([progress_pct, remaining_pct],  # 兩塊：已完成 vs 未完成
               labels=[f'已完成\n{progress_pct:.1f}%', f'未完成\n{remaining_pct:.1f}%'],  # 標籤顯示百分比
               colors=['#2b5c8f', '#d9d9d9'], startangle=90, autopct='%1.1f%%')  # 深藍+淺灰，從90度開始，扇形上標百分比
    ax_pie.set_title("今日生產達標進度", fontsize=12, fontweight='bold')  # 標題粗體

    # 2. 基礎統計數據表
    lengths = df["晶片長度(mm)"].values  # 取出晶片長度欄轉陣列，後續統計用
    summary_data = {  # 組基礎統計表字典
        "指標名稱": ["目前總抽檢數", "晶片長度平均數 (Mean)", "標準差 (Std, Overall)", "最大值 (Max)", "最小值 (Min)"],
        "統計數值": [
            f"{current_qty} 筆",  # 總筆數
            f"{np.mean(lengths):.4f} mm (目標: {target_mean})",  # 平均取4位，並顯示目標方便比對
            f"{np.std(lengths, ddof=1):.4f} (目標: {target_sigma})",  # 整體標準差(n-1)，對比目標sigma
            f"{np.max(lengths):.4f} mm",  # 最大值
            f"{np.min(lengths):.4f} mm"  # 最小值
        ]
    }
    summary_df = pd.DataFrame(summary_data)  # 轉 DataFrame 供 Gradio 表格顯示

    # 3. SPC X-bar & R 管制圖與能力指標計算
    subgroup_size = 5  # 每 5 筆為一子組，對應上面 n=5 常數表
    df["subgroup"] = (np.arange(len(df)) // subgroup_size) + 1  # 依列順序編子組號：0-4→1組，5-9→2組...
    df_spc = df.rename(columns={"晶片長度(mm)": "measurement"})  # 改欄名給 bootstrap 函數用 (它吃 measurement/subgroup)
    
    d2 = d2_TABLE[subgroup_size]  # 查表取 d2=2.326，sigma_within = R-bar/d2
    A2 = A2_TABLE[subgroup_size]  # A2=0.577，X-bar 界限用
    D3 = D3_TABLE[subgroup_size]  # D3=0，R 下界用
    D4 = D4_TABLE[subgroup_size]  # D4=2.114，R 上界用

    # 計算各子組 Mean 與 Range
    subgroup_stats = df_spc.groupby("subgroup")["measurement"].agg(['mean', lambda x: x.max() - x.min()]).reset_index()  # 每組算平均+全距( max-min)
    subgroup_stats.columns = ["subgroup", "x_bar", "r_val"]  # 重命名：子組/組平均/組全距

    x_double_bar = subgroup_stats["x_bar"].mean()  # 總平均 X-double-bar：X-bar 圖中心線
    r_bar = subgroup_stats["r_val"].mean()  # 平均全距 R-bar：R 圖中心線
    sigma_within = r_bar / d2  # 組內標準差估計

    # X-bar 管制界限
    ucl_x = x_double_bar + A2 * r_bar  # X-bar 上界
    lcl_x = x_double_bar - A2 * r_bar  # X-bar 下界
    # R 管制界限
    ucl_r = D4 * r_bar  # R 上界
    lcl_r = D3 * r_bar  # R 下界 (n=5 時為 0)

    # 繪製 SPC 管制圖
    fig_spc, (ax_x, ax_r) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)  # 上下兩圖共用X軸：上=X-bar，下=R
    
    # X-bar 圖
    ax_x.plot(subgroup_stats["subgroup"], subgroup_stats["x_bar"], marker='o', color='#005f73', linewidth=1.5)  # 畫組平均折線深青色圓點
    ax_x.axhline(ucl_x, color='r', linestyle='--', label=f'UCL ({ucl_x:.3f})')  # 上界紅虛線
    ax_x.axhline(x_double_bar, color='g', linestyle='-', label=f'CL ({x_double_bar:.3f})')  # 中心線綠實線
    ax_x.axhline(lcl_x, color='r', linestyle='--', label=f'LCL ({lcl_x:.3f})')  # 下界紅虛線
    ax_x.axhline(usl_val, color='purple', linestyle=':', label=f'USL ({usl_val})')  # 規格上限紫點線，判斷是否偏規格
    ax_x.axhline(lsl_val, color='purple', linestyle=':', label=f'LSL ({lsl_val})')  # 規格下限紫點線
    ax_x.set_title("X-bar 管制圖 (Subgroup 平均值)", fontsize=11, fontweight='bold')
    ax_x.legend(loc='upper right', fontsize=8)  # 圖例右上小字
    ax_x.grid(True, alpha=0.3)  # 淡格線

    # R 圖
    ax_r.plot(subgroup_stats["subgroup"], subgroup_stats["r_val"], marker='s', color='#ca6702', linewidth=1.5)  # 畫全距折線橘色方點
    ax_r.axhline(ucl_r, color='r', linestyle='--', label=f'UCL ({ucl_r:.3f})')  # 上界
    ax_r.axhline(r_bar, color='g', linestyle='-', label=f'CL ({r_bar:.3f})')  # 中心線
    ax_r.axhline(lcl_r, color='r', linestyle='--', label=f'LCL ({lcl_r:.3f})')  # 下界
    ax_r.set_title("R 管制圖 (Subgroup 全距)", fontsize=11, fontweight='bold')
    ax_r.set_xlabel("子組編號 (Subgroup No.)")  # X軸：子組順序
    ax_r.legend(loc='upper right', fontsize=8)
    ax_r.grid(True, alpha=0.3)
    plt.tight_layout()  # 自動調邊距

    # 計算 Cp, Cpk, Pp, Ppk 及 Bootstrap CI
    cap_res = capability_metrics(lengths, sigma_within, lsl_val, usl_val)  # 呼叫第1節函數算能力指標
    boot_ci = bootstrap_capability(df_spc, lsl_val, usl_val, d2, reps=BOOTSTRAP_REPS)  # 子組重抽樣估 95% 區間

    cap_df = pd.DataFrame([  # 組能力指標表：估計值+bootstrap上下限
        {"指標": "Cp (組內潛能)", "估計值": round(cap_res["Cp"], 3), "Bootstrap 95% 下限": round(boot_ci["Cp"][0], 3), "Bootstrap 95% 上限": round(boot_ci["Cp"][1], 3)},
        {"指標": "Cpk (組內能力)", "估計值": round(cap_res["Cpk"], 3), "Bootstrap 95% 下限": round(boot_ci["Cpk"][0], 3), "Bootstrap 95% 上限": round(boot_ci["Cpk"][1], 3)},
        {"指標": "Pp (整體效能)", "估計值": round(cap_res["Pp"], 3), "Bootstrap 95% 下限": round(boot_ci["Pp"][0], 3), "Bootstrap 95% 上限": round(boot_ci["Pp"][1], 3)},
        {"指標": "Ppk (整體能力)", "估計值": round(cap_res["Ppk"], 3), "Bootstrap 95% 下限": round(boot_ci["Ppk"][0], 3), "Bootstrap 95% 上限": round(boot_ci["Ppk"][1], 3)},
        {"指標": "預估總不良率 (PPM)", "估計值": round(cap_res["ppm_total"], 1), "Bootstrap 95% 下限": "-", "Bootstrap 95% 上限": "-"}  # ppm 不給區間故填 -
    ])

    # 4. 鍋爐溫度趨勢與警示燈
    latest_temp = df["鍋爐溫度(°C)"].iloc[-1]  # 取最後一筆即時溫度
    if latest_temp > temp_limit:  # 超過安全上限
        temp_alarm = f"🚨 鍋爐溫度異常！當前 {latest_temp:.1f} °C (超過上限 {temp_limit} °C)"  # 紅燈
    else:
        temp_alarm = f"🟢 鍋爐溫度正常 (當前 {latest_temp:.1f} °C / 上限 {temp_limit} °C)"  # 綠燈

    fig_temp, ax_temp = plt.subplots(figsize=(8, 2.5))  # 建溫度趨勢圖
    ax_temp.plot(df.index, df["鍋爐溫度(°C)"], color='#ae2012', label='鍋爐實時溫度 (°C)')  # 紅色折線
    ax_temp.axhline(temp_limit, color='black', linestyle='--', linewidth=1.5, label=f'安全上限 ({temp_limit}°C)')  # 黑虛線為上限
    ax_temp.set_title("鍋爐溫度趨勢監控", fontsize=10, fontweight='bold')
    ax_temp.set_ylabel("溫度 (°C)")
    ax_temp.legend(loc='upper left', fontsize=8)
    ax_temp.grid(True, alpha=0.3)
    plt.tight_layout()

    # 5. 廠房濕度趨勢與警示燈
    latest_hum = df["廠房濕度(%)"].iloc[-1]  # 取最後一筆濕度
    if latest_hum > hum_limit:  # 超標
        hum_alarm = f"🚨 廠房濕度超標！當前 {latest_hum:.1f} % (超過上限 {hum_limit} %)"
    else:
        hum_alarm = f"🟢 廠房濕度正常 (當前 {latest_hum:.1f} % / 上限 {hum_limit} %)"

    fig_hum, ax_hum = plt.subplots(figsize=(8, 2.5))  # 建濕度趨勢圖
    ax_hum.plot(df.index, df["廠房濕度(%)"], color='#005f73', label='廠房實時濕度 (%)')  # 深青色折線
    ax_hum.axhline(hum_limit, color='black', linestyle='--', linewidth=1.5, label=f'濕度上限 ({hum_limit}%)')  # 上限線
    ax_hum.set_title("廠房濕度趨勢監控", fontsize=10, fontweight='bold')
    ax_hum.set_ylabel("濕度 (%)")
    ax_hum.legend(loc='upper left', fontsize=8)
    ax_hum.grid(True, alpha=0.3)
    plt.tight_layout()

    return (current_qty, alarm_light, summary_df, cap_df, fig_pie, fig_spc,  # 回傳10輸出：產量/燈號/統計表/能力表/圓餅/管制圖
            temp_alarm, fig_temp, hum_alarm, fig_hum)  # 溫度燈號/溫度圖/濕度燈號/濕度圖

In [ ]:
# ==========================================
# 3. 建立 Gradio UI 介面 (左側 Panel + 上方警示 + 右側圖表)
# ==========================================
# 本節用 Gradio Blocks 排版：左側參數 Sidebar + 右側警示/圖表/表格，並用 Timer 每秒自動刷新

with gr.Blocks(title="智慧製造 SPC 實時監控與能力分析儀表板") as demo:  # 建 Blocks 容器，title 為瀏覽器分頁標題，demo 為後續 launch 用
    gr.Markdown("# 🏭 智慧製造實時產線 SPC 與環境監控儀表板")  # 頂部大標題

    # 左側控制面板 (Sidebar)
    with gr.Sidebar(label="⚙️ 控制與參數設定"):  # 左側可收合側邊欄，放所有可調參數
        gr.Markdown("### 🎯 生產目標與規格設定")  # 分組標題：規格區
        in_target_qty = gr.Number(value=100, label="今日預計生產目標數量", precision=0)  # 目標產量，整數，算達成率用
        in_lsl = gr.Number(value=2.95, label="晶片長度下限 LSL (mm)")  # 規格下限，傳給 update_dashboard 算 Cpk/畫管制圖
        in_usl = gr.Number(value=3.05, label="晶片長度上限 USL (mm)")  # 規格上限
        in_mean = gr.Number(value=3.00, label="目標中心 Mean (mm)")  # 目標均值，顯示在統計表供比對
        in_sigma = gr.Number(value=0.025, label="標準差 Sigma (mm)")  # 目標 sigma，顯示在統計表供比對

        gr.Markdown("---")  # 分隔線
        gr.Markdown("### ⚠️ 環境預警閾值設定")  # 分組標題：環境閾值區
        in_temp_limit = gr.Number(value=115.0, label="鍋爐溫度警報上限 (°C)")  # 溫度上限，超過亮紅燈
        in_hum_limit = gr.Number(value=80.0, label="廠房濕度警報上限 (%)")  # 濕度上限，超過亮紅燈

    # 右側主畫面：頂部警示欄 + 下方圖表區
    with gr.Column():  # 右側主欄，垂直排列各監控區塊
        # 1. 頂部狀態與警示燈區塊 (Top Alert Panel)
        gr.Markdown("### 🔔 系統實時狀態與異常警報")  # 區塊標題
        with gr.Row():  # 一列放 4 個警示元件，scale 控制寬度比
            out_qty = gr.Number(label="目前已生產總數 (筆)", interactive=False, scale=1)  # 已生產筆數唯讀，窄欄
            out_alarm_light = gr.Textbox(label="產量達標警示", interactive=False, scale=2)  # 產量燈號文字框唯讀，寬欄
            out_temp_alarm = gr.Textbox(label="鍋爐溫度狀態警示", interactive=False, scale=2)  # 溫度燈號
            out_hum_alarm = gr.Textbox(label="廠房濕度狀態警示", interactive=False, scale=2)  # 濕度燈號

        gr.Markdown("---")

        # 2. 進度與基礎統計
        with gr.Row():  # 一列：左圓餅圖 + 右統計表
            out_pie_plot = gr.Plot(label="生產達標圓餅圖", scale=1)  # 顯示 fig_pie 達成率
            out_summary_table = gr.Dataframe(  # 顯示 summary_df 均值/標準差/最大最小
                label="晶片數據基礎統計分析", 
                headers=["指標名稱", "統計數值"],  # 固定兩欄表頭
                scale=2  # 表格較寬
            )

        gr.Markdown("---")

        # 3. SPC 管制圖與能力指標 (含 Bootstrap 95% CI)
        gr.Markdown("### 📊 SPC X-bar & R 管制圖與製程能力指標")  # 區塊標題
        with gr.Row():  # 單列放管制圖 (圖較寬佔滿)
            out_spc_plot = gr.Plot(label="SPC 管制圖 (X-bar & R Chart)")  # 顯示 fig_spc 上下兩圖
        with gr.Row():  # 下一列放能力表
            out_cap_table = gr.Dataframe(label="能力指標 (Cp, Cpk, Pp, Ppk) 與 Bootstrap 95% 信賴區間")  # 顯示 cap_df

        gr.Markdown("---")

        # 4. 設備與環境趨勢圖
        gr.Markdown("### 🌡️ 設備與環境趨勢監控")  # 區塊標題
        with gr.Row():  # 一列並排溫/濕度趨勢圖
            out_temp_plot = gr.Plot(label="鍋爐溫度折線圖")  # 顯示 fig_temp
            out_hum_plot = gr.Plot(label="廠房濕度折線圖")  # 顯示 fig_hum

    # 自動刷新 Timer (每 1 秒自動調用 update_dashboard)
    timer = gr.Timer(1)  # 建立每 1 秒觸發一次的計時器，配合模擬器每秒寫一筆
    
    inputs_list = [in_target_qty, in_lsl, in_usl, in_mean, in_sigma, in_temp_limit, in_hum_limit]  # 打包 7 個輸入，對應 update_dashboard 的 7 個參數順序
    outputs_list = [out_qty, out_alarm_light, out_summary_table, out_cap_table, out_pie_plot, out_spc_plot,  # 打包 10 個輸出
                    out_temp_alarm, out_temp_plot, out_hum_alarm, out_hum_plot]  # 順序必須與 update_dashboard return 的 10 個值一一對應

    # 當 Timer 觸發時自動更新
    timer.tick(fn=update_dashboard, inputs=inputs_list, outputs=outputs_list)  # 每次 tick 自動呼叫函數重讀 Excel 刷新全畫面

# 啟動 Gradio 介面
demo.queue().launch(share=False)  # queue 啟用排隊防併發，share=False 不產生公開連結只跑本地